<a href="https://colab.research.google.com/github/deepan98raj-dotcom/git_notes_/blob/main/Session_46_Finetuning_LLMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## STEP 1: Dependencies Installation (Updated Package Specs)
### Package Installation (`!pip install`)

* **`--no-deps`**: Prevents `pip` from automatically resolving and overriding pre-installed dependency versions, keeping Colab's default environment stable and preventing package conflicts.
* **`unsloth`**: The core library providing optimized Triton kernels for up to 2x faster training and 80% reduced VRAM usage.
* **`xformers`**: Provides memory-efficient attention mechanisms to accelerate transformer operations.
* **`trl` (Transformer Reinforcement Learning)**: Provides the `SFTTrainer` class for Supervised Fine-Tuning.
* **`peft` (Parameter-Efficient Fine-Tuning)**: Manages LoRA adapters and low-rank parameter updates.
* **`accelerate`**: Handles hardware abstraction and optimized execution across GPUs.
* **`bitsandbytes`**: Implements 4-bit and 8-bit quantization algorithms (QLoRA).
* **`datasets`**: Hugging Face library used for loading, parsing, and preprocessing fine-tuning data.
* **`triton`**: PyTorch language and compiler for writing custom high-performance GPU kernels.
* **`unsloth_zoo`**: Helper utility package providing extra utilities and functions optimized for Unsloth.

---

### Module Imports

* **`torch`**: PyTorch base framework for tensor operations and GPU memory management.
* **`FastLanguageModel`**: Primary Unsloth wrapper for loading quantized base models and patching LoRA adapters.
* **`is_bfloat16_supported`**: Utility function to check if the underlying GPU hardware supports `bfloat16` precision (Ampere generation or newer).
* **`SFTTrainer` & `SFTConfig`**: Standard fine-tuning trainer class and configuration objects for structured supervised training runs.
* **`TrainingArguments`**: Configures training hyperparameters (learning rate, batch size, epochs, logging).
* **`notebook_login`**: Provides an interactive login widget to authenticate with your Hugging Face account inside Colab.

In [ ]:
# ==============================================================================
# STEP 1: Dependencies Installation (Updated Package Specs)
# ==============================================================================
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets triton
!pip install --no-deps unsloth_zoo

import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from huggingface_hub import notebook_login

# Model Loading & PEFT/LoRA Setup Explanation

This document explains the code block used for fine-tuning Large Language Models (LLMs) efficiently using **Unsloth** and **PEFT / QLoRA**.

---

## STEP 2: Load Model & Tokenizer

This section handles downloading and loading the foundational pre-trained model with low-memory 4-bit precision.

* **`max_seq_length = 2048`**: Sets the maximum token length the model can process at once (input prompt + output generation).
* **`dtype = None`**: Automatically detects the best precision format for your hardware (e.g., `Float16` for older GPUs or `Bfloat16` for Ampere+ GPUs).
* **`load_in_4bit = True`**: Enables 4-bit quantization (QLoRA). This drastically reduces VRAM usage by up to 80%, enabling fine-tuning on free consumer GPUs (like Colab T4).
* **`FastLanguageModel.from_pretrained(...)`**:
* **`model_name = "unsloth/Llama-3.2-3B-Instruct"`**: Loads Meta's Llama 3.2 3B instruction-tuned model, pre-optimized by Unsloth for faster loading and training.



In [ ]:
# ==============================================================================
# STEP 2: Load Model & Tokenizer
# ==============================================================================
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## STEP 3: Setup PEFT & LoRA Adapters

* **`FastLanguageModel.get_peft_model(...)`**: Wraps the loaded model with LoRA (Low-Rank Adaptation) configuration layers.
* **`r = 16`**: The rank of the LoRA update matrices. A rank of 16 provides a strong balance between keeping trainable parameter count low and maintaining fine-tuning learning capacity.
* **`target_modules`**: Specifies which internal Transformer projections receive LoRA adapters:
  * **`q_proj`, `k_proj`, `v_proj`, `o_proj`**: Self-Attention mechanism modules (Query, Key, Value, Output).
  * **`gate_proj`, `up_proj`, `down_proj`**: Feed-forward network (FFN / MLP) modules.
* **`lora_alpha = 16`**: A scaling factor applied to the LoRA weight updates. With $r=16$ and $\alpha=16$, the scaling ratio ($\frac{\alpha}{r}$) equals 1.
* **`lora_dropout = 0`**: Disables dropout during fine-tuning. Unsloth's custom Triton kernels achieve maximum throughput and efficiency with 0 dropout.
* **`bias = "none"`**: Excludes bias terms from training to conserve GPU memory and accelerate computation.
* **`use_gradient_checkpointing = "unsloth"`**: Uses Unsloth's optimized gradient checkpointing to dramatically lower VRAM consumption by recomputing activations during backward passes.
* **`random_state = 3407`**: Sets a fixed seed for reproducible initialization across runs.

In [ ]:

# ==============================================================================
# STEP 3: Setup PEFT & LoRA Adapters
# ==============================================================================
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## STEP 4: Conversational Dataset Formatting (Modern Unsloth Way)
### Dataset Structure (`dataset_data` & `Dataset.from_list`)

* **`messages`**: Standard OpenAI-style message dictionary structure containing conversation roles:
  * **`system`**: Defines the model's persona, constraints, and instructions.
  * **`user`**: The raw query input sent to the system.
  * **`assistant`**: The expected output target (label) for training supervision.
* **`Dataset.from_list(...)`**: Converts the raw Python list of dictionaries into a Hugging Face `Dataset` object for batch processing.

---

### Chat Template Preprocessing (`get_chat_template` & `apply_chat_template`)

* **`get_chat_template(...)`**: Configures the tokenizer with special tokens (e.g., `<|start_header_id|>`, `<|eot_id|>`) matching the specific model family (`llama-3.1`).
* **`format_prompts(examples)`**: A mapping function that applies the template to each conversation dictionary:
  * **`tokenize = False`**: Outputs raw formatted text strings instead of token IDs so the trainer can tokenize them dynamically during batch preparation.
  * **`add_generation_prompt = False`**: Keeps the assistant's target answer in the string for supervised fine-tuning instead of leaving it open for generation.
* **`dataset.map(..., batched = True)`**: Efficiently processes the dataset in batches, adding a formatted `"text"` column to each sample.

In [ ]:
# ==============================================================================
# STEP 4: Conversational Dataset Formatting (Modern Unsloth Way)
# ==============================================================================
# Direct conversational format (Messages list)
dataset_data = [
    {
        "messages": [
            {"role": "system", "content": "You are an intelligent routing assistant. Classify user query into department. Output ONLY the label."},
            {"role": "user", "content": "I want to know the pricing for your enterprise plan."},
            {"role": "assistant", "content": "Sales"}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are an intelligent routing assistant. Classify user query into department. Output ONLY the label."},
            {"role": "user", "content": "My subscription did not renew automatically."},
            {"role": "assistant", "content": "Billing"}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are an intelligent routing assistant. Classify user query into department. Output ONLY the label."},
            {"role": "user", "content": "The API is throwing 500 internal server error continuously."},
            {"role": "assistant", "content": "Technical Support"}
        ]
    }
]

dataset = Dataset.from_list(dataset_data)

# Modern SFTTrainer automatically handles chat templates if structured properly in 'messages' key.
# Alternatively, use Unsloth's standard formatting helper:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def format_prompts(examples):
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in examples["messages"]]
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched = True)

## STEP 5: SFTTrainer with modern SFTConfig
### SFTTrainer Initialization Arguments

* **`model`**: The quantized base model configured with LoRA adapters.
* **`tokenizer`**: The tokenizer configured with the appropriate chat template.
* **`train_dataset`**: The formatted Hugging Face dataset containing prompt text.
* **`dataset_num_proc = 2`**: Uses 2 CPU worker processes to speed up data preprocessing and batching.
* **`packing = False`**: Disables sample packing, treating each multi-turn chat sample as an individual instance rather than stitching multiple samples into single 2048-token blocks.

---

### SFTConfig Hyperparameters (`args`)

* **`per_device_train_batch_size = 2`**: Processes 2 training samples per GPU step.
* **`gradient_accumulation_steps = 4`**: Accumulates gradients across 4 steps before performing weight updates. This simulates an effective batch size of $2 \times 4 = 8$ without increasing GPU peak VRAM memory.
* **`warmup_steps = 5`**: Gradually increases the learning rate over the first 5 steps to stabilize training and prevent gradient explosion.
* **`max_steps = 60`**: Sets the total number of training iterations to 60.
* **`learning_rate = 2e-4`**: The peak learning rate ($0.0002$) used for weight optimization.
* **`fp16` & `bf16`**: Dynamically toggles 16-bit precision based on GPU hardware support (`bf16` for Ampere / NVIDIA T4 fallback to `fp16`).
* **`logging_steps = 1`**: Logs training metrics (loss, learning rate) at every training step.
* **`optim = "adamw_8bit"`**: Employs an 8-bit quantized AdamW optimizer to significantly cut VRAM usage compared to standard 32-bit AdamW.
* **`weight_decay = 0.01`**: Applies L2 regularization to prevent model weights from growing too large and overfitting.
* **`lr_scheduler_type = "linear"`**: Linearly decays the learning rate to zero after the warmup phase.
* **`dataset_text_field = "text"`**: Points to the target column in the dataset containing the formatted chat string.
* **`max_seq_length = max_seq_length`**: Limits inputs to a maximum length of 2048 tokens.

---

### Training Execution

* **`trainer_stats = trainer.train()`**: Executes the training loop, updating LoRA adapter parameters and returning detailed performance statistics (e.g., runtime, loss history, steps per second).

In [ ]:
# ==============================================================================
# STEP 5: SFTTrainer with modern SFTConfig
# ==============================================================================
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2, #2 prompt/sample per step
        gradient_accumulation_steps = 4, # 4 Steps before backpropagation
        warmup_steps = 5, # Avoids Models Explosion gradient
        max_steps = 60, #Training steps
        learning_rate = 2e-4, #Used in updating weight
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", #Optimization
        weight_decay = 0.01, #Avoids Overfitting
        lr_scheduler_type = "linear", #Step by step linear type after conversion
        seed = 3407,
        output_dir = "outputs",
        dataset_text_field = "text",
        max_seq_length = max_seq_length, #Total 2048 length
    ),
)

trainer_stats = trainer.train()


## STEP 6: Fast Inference Setup
### Inference Optimization

* **`FastLanguageModel.for_inference(model)`**: Activates Unsloth's fast inference mode (enabling optimized GPU kernels and KV caching for up to 2x faster token generation).

---

### Prompt Tokenization (`apply_chat_template`)

* **`messages`**: The sample evaluation conversation containing the `system` role instructions and a new `user` query.
* **`tokenize = True`**: Immediately converts the formatted chat template string into numerical token IDs.
* **`add_generation_prompt = True`**: Appends the assistant turn delimiter (e.g., `<|start_header_id|>assistant<|end_header_id|>`) to signal the model to start generating its output.
* **`return_tensors = "pt"`**: Returns PyTorch tensors instead of standard Python lists.
* **`.to("cuda")`**: Transfers the input tensors to GPU memory for inference.

---

### Generation Engine (`model.generate`)

* **`input_ids = inputs`**: Passes the tokenized prompt matrix to the model.
* **`max_new_tokens = 64`**: Limits generation length to a maximum of 64 new tokens (ideal for concise classification labels).
* **`use_cache = True`**: Reuses computed Key-Value (KV) attention states from previous tokens to significantly accelerate decoding.
* **`temperature = 0.1`**: Sets a very low sampling temperature to force highly deterministic, precise, and consistent outputs.

---

### Decoding & Formatting (`tokenizer.batch_decode`)

* **`skip_special_tokens=True`**: Strips structural chat tags (like `<|eot_id|>`) from the final string output.
* **`response[0]`**: Extracts and prints the clean response string for the first batch sample (expected output: `"Sales"`).

In [ ]:
# ==============================================================================
# STEP 6: Fast Inference Setup
# ==============================================================================
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are an intelligent routing assistant. Classify user query into department. Output ONLY the label."},
    {"role": "user", "content": "We are looking to buy a license for an entire engineering team."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 64,
    use_cache = True,
    temperature = 0.1
)

response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print("--- MODEL OUTPUT RESPONSE ---")
print(response[0])

### 1. Local Merged Model Export (`save_pretrained_merged`)

* **`save_directory` (`"llama_3_2_ticket_classifier"`)**: Local folder path where the merged model architecture and tokenizer files will be written.
* **`tokenizer`**: Exports tokenizers with matching special token mapping alongside the model weights.
* **`save_method = "merged_16bit"`**: Merges the LoRA adapter weights directly back into the base 16-bit model layers. This creates a standalone model suitable for deployment in vLLM, Hugging Face `transformers`, or TGI without needing extra adapter loading steps.

---

### 2. GGUF Quantization & HF Upload (`push_to_hub_gguf`)

* **`repo_name_gguf`**: The target Hugging Face Hub repository (e.g., `username/llama-3.2-3b-ticket-classifier-gguf`).
* **`quantization_method = "q4_k_m"`**: Converts weights to GGUF format using 4-bit Medium K-quantization (`q4_k_m`). This provides optimal accuracy retention while running efficiently in llama.cpp, Ollama, LM Studio, or local CPU/GPU runtimes.

In [ ]:
# ==============================================================================
# STEP 7: Export Model
# ==============================================================================
# Unsloth export APIs remain clean & efficient:
# model.save_pretrained_merged("llama_3_2_ticket_classifier", tokenizer, save_method = "merged_16bit")
# model.push_to_hub_gguf("repo_name_gguf", tokenizer, quantization_method = "q4_k_m")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-b53l5l2k/unsloth_57280c8f4ce044389b4648c8ddbd27ce
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-b53l5l2k/unsloth_57280c8f4ce044389b4648c8ddbd27ce
  Resolved https://github.com/unslothai/unsloth.git to commit fe7452025f939d9c306a834bcf8855262ba91f60
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.9.4-py3-none-any.whl size=7733444 sha256=49a824c005903e93432ec00556643618284280b36309fe91a2cc7983ecca8ca3
  Stored in directory: /tmp/pip-ephem-wheel-cache-y1v2m32u/wheels/d5/36/1d/4e65996c5b80c84a5ac1b0ba10718bdc155f8dd04352746a8f
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,4.914362
2,4.914362
3,4.748073
4,4.219140
5,3.687503
6,3.178775
7,2.689757
8,2.234346
9,1.853074
10,1.434801


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- MODEL OUTPUT RESPONSE ---
system

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are an intelligent routing assistant. Classify user query into department. Output ONLY the label.user

We are looking to buy a license for an entire engineering team.assistant

Sales


### Function Logic (`route_ticket`)

* **`FastLanguageModel.for_inference(model)`**: Switches internal model layers to optimized evaluation mode.
* **`do_sample = False`**: Disables randomized sampling entirely, forcing greedy search for deterministic, reproducible classification outputs.
* **`outputs[0][inputs.shape[-1]:]`**: Slices the output tensor to extract **only the newly generated response tokens**, removing the prompt input tokens before decoding.
* **`tokenizer.decode(..., skip_special_tokens=True)`**: Converts predicted token IDs into a clean string without system chat tags.
* **`.strip()`**: Removes leading/trailing whitespaces or extra line breaks around the label.

---

### Live Batch Test (`test_queries`)

* **`test_queries`**: Evaluates three real-world customer support scenarios:
  * *"I need to upgrade from my trial to a paid seat for 5 people."* $\rightarrow$ **Sales**
  * *"The website keeps logging me out and giving me error code 403."* $\rightarrow$ **Technical Support**
  * *"Can you update the credit card on file for our monthly invoice?"* $\rightarrow$ **Billing**
* **`for query in test_queries:`**: Loops through each test query sequentially and prints the target query alongside its predicted department label.

In [ ]:
# ==============================================================================
# DEMO CELL: Interactive Ticket Routing Tester
# ==============================================================================
def route_ticket(user_query: str):
    # Ensure model is in inference mode
    FastLanguageModel.for_inference(model)

    messages = [
        {"role": "system", "content": "You are an intelligent routing assistant. Classify user query into department. Output ONLY the label."},
        {"role": "user", "content": user_query}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 16,
        use_cache = True,
        do_sample = False, # Replaces temperature = 0.0 for deterministic output
    )

    # Decode only the generated response
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response.strip()

# --- Live Classroom Demo Execution ---
test_queries = [
    "I need to upgrade from my trial to a paid seat for 5 people.",
    "The website keeps logging me out and giving me error code 403.",
    "Can you update the credit card on file for our monthly invoice?"
]

print("=== LIVE MODEL ROUTING DEMO ===")
for query in test_queries:
    department = route_ticket(query)
    print(f"\nUser Query: \"{query}\"")
    print(f"--> Predicted Department: [{department}]")

Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== LIVE MODEL ROUTING DEMO ===

User Query: "I need to upgrade from my trial to a paid seat for 5 people."
--> Predicted Department: [Sales]


Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



User Query: "The website keeps logging me out and giving me error code 403."
--> Predicted Department: [Technical Support]


Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



User Query: "Can you update the credit card on file for our monthly invoice?"
--> Predicted Department: [Billing]


### Inference Helper (`route_ticket`)

* **`FastLanguageModel.for_inference(model)`**: Switches the model to optimized inference mode for faster generation.
* **`messages`**: Formats the dynamic runtime query into an OpenAI-style chat payload containing system instructions and user input.
* **`tokenizer.apply_chat_template(..., tokenize=True)`**: Tokenizes the formatted conversation array into GPU tensors.
* **`do_sample=False`**: Enforces deterministic, greedy search decoding for consistent classification outputs.
* **`outputs[0][inputs.shape[-1]:]`**: Slices out input prompt tokens to decode **only** the model's new completion text.
* **`.strip()`**: Cleans up leading/trailing whitespace around the department label.

---

### Simulation Loop (`run_simulation`)

* **`while True:`**: Runs an interactive loop to continuously accept and process user queries.
* **`except (EOFError, KeyboardInterrupt):`**: Gracefully handles manual interrupts (e.g., `Ctrl+C`) or closed stdin streams without crashing the notebook kernel.
* **Empty Input Check (`if not user_query:`):** Prevents submitting blank strings to the model and prompts for a valid entry.
* **Exit Handler (`user_query.lower() in ("exit", "quit")`):** Safely breaks out of the loop when key phrases are entered.
* **Error Catching (`try...except Exception`):** Catches and displays inference runtime errors (such as GPU OOM) without abruptly terminating the loop.

In [ ]:
# ==============================================================================
# INTERACTIVE TICKET ROUTING SIMULATION
# ==============================================================================

def route_ticket(user_query: str) -> str:
    """

    using finetuned model users query is classified in to dept
    """
    FastLanguageModel.for_inference(model)

    messages = [
        {
            "role": "system",
            "content": "You are an intelligent routing assistant. Classify user query into department. Output ONLY the label."
        },
        {"role": "user", "content": user_query}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=16,
        use_cache=True,
        do_sample=False,
    )


    #Skipping the prompt and decoding the generated respons
    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )
    return response.strip()


def run_simulation():
    """
    User-Getting query from User and predicting the department
    """
    print("=" * 60)
    print("TICKET ROUTING SIMULATION")
    print("=" * 60)
    print("Type your customer support query below.")
    print("Type 'exit' or 'quit' to stop the simulation.")
    print("=" * 60)

    while True:
        try:
            user_query = input("\nEnter your query: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSimulation stopped.")
            break

        # Empty input check
        if not user_query:
            print("Please enter a valid query.")
            continue

        # Exit condition
        if user_query.lower() in ("exit", "quit"):
            print("Simulation ended. Goodbye.")
            break

        # Predict department
        try:
            department = route_ticket(user_query)
            print(f"Predicted Department: {department}")
        except Exception as e:
            print(f"Error during prediction: {e}")


# Run the simulation
run_simulation()

TICKET ROUTING SIMULATION
Type your customer support query below.
Type 'exit' or 'quit' to stop the simulation.

Enter your query: my refund has not been credited as promised yet . When will you process me ?


Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Predicted Department: Sales

Enter your query: My router password has been resetted without my knowledge . Kindly help me fixing the passowrd


Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Predicted Department: Technical Support
